In [ ]:
import numpy as np
import pandas as pd

from gamblersruin.data import attach_odds, fetch_history, fetch_odds, missing_odds
from gamblersruin.evaluation import evaluate_season, results_table, rps_each
from gamblersruin.strength import StrengthConfig

LEAGUE = "ENG Premier League"
WARMUP = "2021-2022"
SEASONS = ["2022-2023", "2023-2024", "2024-2025", "2025-2026", "2026-2027"]
all_seasons = [WARMUP, *SEASONS]

history = attach_odds(fetch_history(LEAGUE, all_seasons), fetch_odds(LEAGUE, all_seasons))
print(history[history.season.isin(SEASONS)].groupby("season").size().to_dict(), "played matches")
print(
    f"{missing_odds(history[history.season.isin(SEASONS)]).sum()} played match(es) without odds are left out of the scoring")

configs = {f"latest {n}": StrengthConfig(recent_matches=n) for n in (5, 10, 15, 20, 30)}

evaluated = {
    season: evaluate_season(history, season, prior_season, configs)
    for prior_season, season in zip(all_seasons, all_seasons[1:])
}

table = results_table(evaluated)
table.sort_values(table.columns[-1]).round(4)

In [ ]:
MODEL, MARKET, CLOSING = "latest 5", "Bookmaker (market average)", "Bookmaker (market average, closing)"


def short(season):
    return f"{season[:4]}-{season[-2:]}"


def gap(model, market):
    diff = model - market
    return diff.mean(), 1.96 * diff.std(ddof=1) / np.sqrt(len(diff))


groups = {short(s): [s] for s in SEASONS} | {
    f"{short(SEASONS[0])} to {short(SEASONS[1])}": SEASONS[:2],
    f"{short(SEASONS[2])} to {short(SEASONS[-1])}": SEASONS[2:],
    "All seasons": SEASONS,
}

rows = []
for label, seasons in groups.items():
    rps = {
        name: np.concatenate([rps_each(evaluated[s].predictions[name], evaluated[s].outcome) for s in seasons])
        for name in (MARKET, CLOSING, MODEL, "League frequencies")
    }
    rows.append([
        label, len(rps[MODEL]), *(r.mean() for r in rps.values()),
        *gap(rps[MODEL], rps[MARKET]), *gap(rps[MODEL], rps[CLOSING]),
    ])

columns = ["Season", "Matches", "Market", "Closing", "Model", "League frequencies", "Gap", "95% ±", "Gap (closing)",
           "95% ±"]
pd.DataFrame(rows, columns=columns).set_index("Season").round(4)